<a href="https://colab.research.google.com/github/sharathnallaganti/AI_Security/blob/main/AgentGuardipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1: Environment Setup & Project Skeleton

# AgentGuard – Stage 1: Setup
- Install required libraries
- Create project structure
- Define basic configuration



In [6]:
!pip install -q langchain langchain-openai openai python-dotenv gradio

import os
from pathlib import Path

# Create project folders
folders = ["agent", "middleware", "policies", "detectors", "logs", "demo"]
for folder in folders:
    Path(folder).mkdir(exist_ok=True)

print("Project folders created")

Project folders created


#  Stage 2: Build a Simple AI Agent with ToolsGoal:

Create a basic agent that can use tools (so we have something to protect).

Simple Agent + Tools

Create mock tools (file read, email, shell, search) and a basic agent that can call them.

In [7]:
from langchain.tools import tool

@tool
def read_file(path: str) -> str:
    """Read a file from the system"""
    return f"[MOCK] Content of {path}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email"""
    return f"[MOCK] Email sent to {to}"

@tool
def run_command(cmd: str) -> str:
    """Run a shell command"""
    return f"[MOCK] Executed: {cmd}"

tools = [read_file, send_email, run_command]
print("Tools ready:", [t.name for t in tools])

Tools ready: ['read_file', 'send_email', 'run_command']


# Stage 3: AgentGuard Middleware (Interceptor Core)Goal:

Build the core middleware that intercepts every tool call.Markdown

Stage 3: Middleware Interceptor

This is the heart of AgentGuard. All tool calls must pass through this layer

In [8]:
class AgentGuardMiddleware:
    def __init__(self):
        self.logs = []

    def intercept(self, tool_name: str, arguments: dict) -> dict:
        """Main interception point"""
        decision = {
            "tool": tool_name,
            "arguments": arguments,
            "action": "allow",   # default
            "reason": "Passed basic check",
            "risk_score": 0.1
        }
        self.logs.append(decision)
        return decision

guard = AgentGuardMiddleware()
print("Middleware ready")

Middleware ready


# Stage 4: Policy Engine

**Goal:** Add rules that decide whether a tool call is allowed.

Define allowed tools, blocked patterns, and risk-based decisions.

In [9]:
class PolicyEngine:
    def __init__(self):
        self.allowed_tools = ["read_file", "send_email", "run_command"]
        self.blocked_patterns = ["rm -rf", "format", "delete", "drop"]

    def evaluate(self, tool_name: str, arguments: dict) -> dict:
        if tool_name not in self.allowed_tools:
            return {"action": "block", "reason": "Tool not allowed", "risk_score": 0.9}

        args_str = str(arguments).lower()
        for pattern in self.blocked_patterns:
            if pattern in args_str:
                return {"action": "block", "reason": f"Dangerous pattern: {pattern}", "risk_score": 0.95}

        return {"action": "allow", "reason": "Policy passed", "risk_score": 0.1}

policy = PolicyEngine()

# Stage 5: Threat Detectors

**Goal:** Add detectors for prompt injection, secrets, and dangerous commands.

In [10]:
import re

class ThreatDetectors:
    def check_prompt_injection(self, text: str) -> bool:
        patterns = [
            r"ignore (all|previous|above) instructions",
            r"system prompt",
            r"you are now",
            r"disregard"
        ]
        return any(re.search(p, text, re.I) for p in patterns)

    def check_secrets(self, text: str) -> bool:
        patterns = [r"sk-", r"api[_-]?key", r"password\s*=", r"secret"]
        return any(re.search(p, text, re.I) for p in patterns)

detectors = ThreatDetectors()

# Stage 6: Audit Logging + Decision Integration

**Goal:** Combine middleware + policy + detectors and log every decision.

Integrate everything and store audit logs.

In [11]:
import json
from datetime import datetime

class AgentGuard:
    def __init__(self):
        self.policy = PolicyEngine()
        self.detectors = ThreatDetectors()
        self.logs = []

    def process_tool_call(self, tool_name: str, arguments: dict):
        args_text = str(arguments)

        # Policy check
        policy_result = self.policy.evaluate(tool_name, arguments)

        # Detector checks
        injection = self.detectors.check_prompt_injection(args_text)
        secrets = self.detectors.check_secrets(args_text)

        action = policy_result["action"]
        reason = policy_result["reason"]
        risk = policy_result["risk_score"]

        if injection:
            action = "block"
            reason = "Prompt injection detected"
            risk = 0.99
        if secrets:
            action = "block"
            reason = "Possible secret leakage"
            risk = 0.98

        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "tool": tool_name,
            "arguments": arguments,
            "action": action,
            "reason": reason,
            "risk_score": risk
        }
        self.logs.append(log_entry)
        return log_entry

guard = AgentGuard()

# Stage 7: Demo Interface + Attack Scenarios

**Goal:** Build a simple interactive demo (Gradio works great in Colab) and test attack scenarios.

In [12]:
import gradio as gr

def demo_agent_call(tool_name, arguments_json):
    try:
        args = json.loads(arguments_json)
    except:
        args = {"raw": arguments_json}

    result = guard.process_tool_call(tool_name, args)
    return json.dumps(result, indent=2)

iface = gr.Interface(
    fn=demo_agent_call,
    inputs=[
        gr.Dropdown(["read_file", "send_email", "run_command"], label="Tool"),
        gr.Textbox(label="Arguments (JSON)", value='{"path": "notes.txt"}')
    ],
    outputs="text",
    title="AgentGuard Demo",
    description="Test tool calls through the security layer"
)

iface.launch(share=True)   # share=True gives a public link

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://00488433c24803d771.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
